# Lab 01-04 — Structure-preserving HTML chunking: split on the heading tags

**Track 01 · Chunking** — HTML documents already carry their own section hierarchy in the heading tags (`<h1>` … `<h4>`). Instead of splitting on a fixed character/token budget and hoping a section boundary lands inside a chunk, `HTMLHeaderTextSplitter` splits *on the headings themselves*: every chunk maps to one document section, and the heading chain that leads to it is stored as chunk metadata.

This notebook is **self-contained**: it imports the LangChain HTML splitter directly — no repo component library. Every block of the pipeline is built right here: the raw HTML read, the heading-aware splitter, and the metadata inspection all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```
HTML heading hierarchy  ->  chunk metadata  ->  section-scoped retrieval
Data: Data/SD-06-tables/aapl-20230930.htm (a real SEC 10-K filing, inline-XBRL)
Split: HTMLHeaderTextSplitter(headers_to_split_on=[h1, h2, h3])
```

A retriever can then filter on `metadata["H2"] == "Liquidity and Capital Resources"` instead of hoping the right text happens to be in the top-k.

Real-world HTML is the point of this lab: SEC filings style their headings as bold `<span>` elements, not `<h1>`-`<h4>` tags, so the splitter finds no heading chain and collapses the whole filing into a single chunk with empty metadata. The printed comparison shows exactly where the `<title>` and the styled top heading end up — and why structure-preserving parsing for these documents needs XBRL/table-aware parsing (the SD-06 track).


## Setup

One prerequisite must hold before this notebook will run:

- **the SEC 10-K sample on disk** — `Data/SD-06-tables/aapl-20230930.htm` (a ~1.5MB real Apple 10-K filing, already fetched by the repo's manifest-verified fetchers).

No repo imports are needed: everything this notebook uses comes from `langchain-core` and `langchain-text-splitters`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q langchain-core langchain-text-splitters


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
from collections import Counter
from pathlib import Path

# LangChain HTML splitter — the only library this notebook needs. Nothing is
# imported from the repo's src/ component library.
from langchain_core.documents import Document  # noqa: E402
from langchain_text_splitters import HTMLHeaderTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

`HTML_PATH` points at the sample file; `HEADERS_TO_SPLIT_ON` is the heading chain the splitter cuts on (h1-h3); `PREVIEW_CHARS` caps the content previews. The lab deliberately runs the splitter on a *real* SEC 10-K filing — real-world HTML is the point.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — the heading chain we split on, and the sample file
# --------------------------------------------------------------------------
HTML_PATH = "Data/SD-06-tables/aapl-20230930.htm"
HEADERS_TO_SPLIT_ON = [("h1", "H1"), ("h2", "H2"), ("h3", "H3")]
PREVIEW_CHARS = 200


## 2. Load + split — raw HTML inline, split on heading tags

`load_raw_html` reads the raw HTML file as text — plain `open()` on purpose: the splitter does the parsing, so the lab never touches the markup itself. `build_splitter` configures `HTMLHeaderTextSplitter` for the h1-h3 heading chain. `section_counts` counts chunks per section, keyed by the metadata value for a given level, with chunks that carry no value for that level (no matching heading tag in the document) falling into the `"(no <tag> found)"` bucket. `preview` collapses whitespace and caps a content preview.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load + split — raw HTML inline, split on heading tags
# --------------------------------------------------------------------------
def load_raw_html(path: str) -> str:
    """Read the raw HTML file as text.

    Plain ``open()`` on purpose: the splitter does the parsing, so the lab
    never touches the markup itself.
    """
    with open(path, encoding="utf-8") as f:
        return f.read()


def build_splitter() -> HTMLHeaderTextSplitter:
    """Build the splitter configured for the h1-h3 heading chain."""
    return HTMLHeaderTextSplitter(headers_to_split_on=HEADERS_TO_SPLIT_ON)


def section_counts(chunks: list[Document], level: str) -> Counter[str]:
    """Count chunks per section, keyed by the metadata value for ``level``.

    Chunks with no value for that level (no matching heading tag in the
    document) fall into the ``"(no <tag> found)"`` bucket.
    """
    return Counter(
        chunk.metadata.get(level, f"(no <{level.lower()}> found)") for chunk in chunks
    )


def preview(text: str, limit: int = PREVIEW_CHARS) -> str:
    """Collapse whitespace and cap a content preview at ~``limit`` chars."""
    return " ".join(text.split())[:limit]


## 3. Experiment — a real SEC 10-K filing

`run_experiment` loads the raw filing and splits it with `HTMLHeaderTextSplitter`. The teaching point is that SEC filings style their headings as bold `<span>` elements, not `<h1>`-`<h4>` tags — so the splitter finds no heading chain and collapses the whole filing into a single chunk with empty metadata. The experiment captures everything the demo prints: the chunk distribution across H1/H2 sections (which buckets into `"(no <h1> found)"`), the metadata chain of the first chunks, the content preview, and where the `<title>` text and the styled top heading (`UNITED STATES / SECURITIES AND EXCHANGE COMMISSION / FORM 10-K`) end up.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — run the splitter on a real SEC 10-K filing
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    # --- load --------------------------------------------------------------
    html = load_raw_html(HTML_PATH)

    # --- split -------------------------------------------------------------
    chunks = build_splitter().split_text(html)

    # --- where the styled top heading ends up ------------------------------
    top = -1
    if chunks:
        top = chunks[0].page_content.find("UNITED STATES")

    return {
        "html_bytes": len(html),
        "chunks": chunks,
        "top": top,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact in the lab's order: the raw HTML size; the split call with its chunk count; the chunk distribution across H1/H2 sections (from chunk metadata); the metadata chain of the first few chunks; the content preview of chunk 0; the comparison of where the `<title>` and the styled top heading went; and the takeaway.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    chunks = exp["chunks"]

    # --- load --------------------------------------------------------------
    print(f"[1] Loaded raw HTML from {HTML_PATH} ({exp['html_bytes']:,} bytes)")

    # --- split -------------------------------------------------------------
    print(
        "[2] Split with HTMLHeaderTextSplitter("
        f"headers_to_split_on={HEADERS_TO_SPLIT_ON})"
    )
    print(f"    -> {len(chunks)} chunk(s)")

    # --- inspect metadata --------------------------------------------------
    print("\n[3] Chunk distribution across H1/H2 sections (from chunk metadata):")
    for level in ("H1", "H2"):
        counts = section_counts(chunks, level)
        print(f"    metadata[{level!r}]:")
        for section, count in counts.most_common():
            print(f"      {section!r}: {count} chunk(s)")

    print("\n[4] Metadata chain of the first few chunk(s):")
    for i, chunk in enumerate(chunks[:3]):
        print(f"    chunk {i}: metadata={chunk.metadata}")

    # --- print: content preview -------------------------------------------
    print("\n[5] Content preview (first ~200 chars of chunk 0):")
    if chunks:
        print(f"    {preview(chunks[0].page_content)}...")

    # --- print: comparison of <title> / top heading ------------------------
    print("\n[6] Where did the section headings go? (comparison)")
    print(
        "    - <title>aapl-20230930</title>: NOT a header (not in "
        "headers_to_split_on) and its text is dropped from the chunks "
        "entirely — the <head> is page metadata, not content."
    )
    if chunks and exp["top"] != -1:
        content = chunks[0].page_content
        top = exp["top"]
        print(
            "    - Top heading 'UNITED STATES / SECURITIES AND EXCHANGE "
            "COMMISSION / FORM 10-K' is a styled <span> (font-weight:700), "
            "NOT an <h1> tag -> it survives as plain text inside the single "
            "chunk but never becomes metadata."
        )
        print(f"      preview around it: {preview(content[top:top + 300])}...")

    # --- takeaway ----------------------------------------------------------
    print(
        "\n[7] Takeaway: HTML heading hierarchy becomes chunk metadata -> "
        "section-scoped retrieval — but only when the document uses real "
        "heading tags. SEC XBRL filings style headings as spans, so "
        "HTMLHeaderTextSplitter collapses the whole filing into one chunk "
        "with no metadata; structure-preserving parsing for these documents "
        "needs XBRL/table-aware parsing (the SD-06 track)."
    )


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: the whole filing collapses into exactly ONE chunk; that chunk carries EMPTY metadata (no heading chain found); the `<title>` text is dropped from the chunk content entirely; and the styled top heading survives as plain text inside the chunk. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    chunks = exp["chunks"]

    checks.append((
        "whole filing collapses into one chunk (no real heading tags)",
        len(chunks) == 1,
    ))
    checks.append((
        "chunk carries empty metadata (no heading chain found)",
        chunks[0].metadata == {},
    ))
    checks.append((
        "every chunk falls in the '(no <h1> found)' bucket",
        section_counts(chunks, "H1")["(no <h1> found)"] == len(chunks),
    ))
    checks.append((
        "every chunk falls in the '(no <h2> found)' bucket",
        section_counts(chunks, "H2")["(no <h2> found)"] == len(chunks),
    ))
    checks.append((
        "<title> text is dropped from the chunk content",
        "aapl-20230930" not in chunks[0].page_content,
    ))
    checks.append((
        "styled top heading survives as plain text inside the chunk",
        exp["top"] != -1,
    ))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A couple of seconds: one raw HTML read (~1.5MB) and a heading-tag split — no downloads, no API calls, no embeddings. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

What happens when structure-preserving splitting meets HTML that does not use real heading tags: one chunk, empty metadata, `<title>` dropped — and the styled top heading surviving only as plain text.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the SEC filing sample is intact.


In [ ]:
verify_gate(exp)
